# Trajectory Scoring: Evaluating the Path, Not the Answer

Based on: [TRACE: Trajectory-Aware Comprehensive Evaluation](https://arxiv.org/abs/2602.21230) (Feb 2026)

## The Problem

Two agents answer "Find flights NYC to London and check the weather":

| | Agent A (Efficient) | Agent B (Wasteful) |
|--|---|---|
| Step 1 | `search_flights("NYC", "London")` | `search_flights("NYC", "London")` |
| Step 2 | `get_weather("London")` | `get_currency_exchange("USD", "GBP")` ← irrelevant |
| Step 3 | Returns answer | `search_flights("NYC", "London")` ← duplicate |
| Step 4 | | `get_weather("London")` |
| Step 5 | | Returns same answer |
| **Result** | ✅ Correct, 2 calls | ✅ Correct, 4 calls (2 wasted) |

Both produce the **same correct answer**, but Agent B wasted tokens on irrelevant and duplicate tool calls. Output-only evaluation scores them equally. **Trajectory evaluation catches the difference.**

## What We Compare

| Approach | What It Measures | Catches Agent B's Issues? |
|----------|-----------------|--------------------------|
| `OutputEvaluator` (output only) | Is the final answer good? | ❌ No |
| `TrajectoryEvaluator` (path) | Did the agent use the right tools in the right order? | ✅ Yes |
| `TrajectoryPlugin` (hooks) | Full timing, failures, duplicates | ✅ Yes + metrics |

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Setup: Simulated agent trajectories

**What this does:** Creates two simulated trajectories — Agent A (efficient, 2 tool calls) and Agent B (wasteful, 4 tool calls) — that both produce the same final answer.

**Why we simulate:** By controlling the trajectories manually, we can guarantee that the only difference is the path, not the output. This isolates what each evaluation approach can and cannot detect.

> **What to look for:** Both agents return the identical answer text. The difference is purely in the tool call sequence: Agent A calls only relevant tools, while Agent B adds an irrelevant call (`get_currency_exchange`) and a duplicate (`search_flights` twice).

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

QUESTION = "Find flights from NYC to London and check the weather there"
SAME_ANSWER = (
    "Flights: BA117 7PM-7AM $450, DL1 9:30PM-9:30AM $520. "
    "London weather: 18C, partly cloudy, 20% rain."
)

# Agent A: efficient — only calls relevant tools
TRAJECTORY_A = [
    {"name": "search_flights"},
    {"name": "get_weather"},
]

# Agent B: wasteful — irrelevant + duplicate calls
TRAJECTORY_B = [
    {"name": "search_flights"},
    {"name": "get_currency_exchange"},  # irrelevant
    {"name": "search_flights"},          # duplicate
    {"name": "get_weather"},
]

print("Agent A trajectory:", [t["name"] for t in TRAJECTORY_A])
print("Agent B trajectory:", [t["name"] for t in TRAJECTORY_B])
print(f"\nBoth return the same answer: '{SAME_ANSWER[:60]}...'")

---
## Test 1: Output-Only Evaluation (blind to trajectory)

**What this does:** Runs `OutputEvaluator` on both agents' answers. This evaluator only sees the final text — it has no access to which tools were called.

**Why this matters:** This is the baseline. Most evaluation frameworks default to output-only scoring, which is insufficient for agent quality. This test demonstrates the blind spot.

> **What to look for:** Both agents should receive nearly identical scores (both high), because their answers are the same text. If both score the same, that confirms output-only evaluation cannot distinguish efficient from wasteful agent behavior.

In [ ]:
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

MODEL = "gpt-4o-mini"

output_eval = OutputEvaluator(
    rubric="Rate helpfulness 0-1. Should include flight details and weather.",
    model=MODEL,
)

cases = [
    Case(name="agent_a", input=QUESTION, expected_output="Flights and weather info"),
    Case(name="agent_b", input=QUESTION, expected_output="Flights and weather info"),
]

# Both return the SAME answer
def output_task(case):
    return SAME_ANSWER

print("=" * 60)
print("TEST 1: OUTPUT-ONLY — Can it tell Agent A from Agent B?")
print("=" * 60)

output_exp = Experiment(cases=cases, evaluators=[output_eval])
output_reports = output_exp.run_evaluations(output_task)
output_reports[0].display()

print("\n⚠️  Both agents scored the same! Output-only evaluation is blind to trajectory quality.")

---
## Test 2: Trajectory Evaluation (sees the path)

**What this does:** Runs `TrajectoryEvaluator` on both agents. This evaluator receives the tool call sequence and scores it against a rubric that penalizes irrelevant tools, duplicates, and illogical order.

**Why trajectory evaluation matters:** The same correct answer can come from a 2-step efficient path or a 10-step wasteful path. In production, wasteful paths mean higher costs, longer latency, and possible reasoning loops. Trajectory evaluation catches these issues that output-only evaluation misses.

> **What to look for:** Agent A should score significantly higher than Agent B. The score gap tells you how much the evaluator penalizes inefficiency. Agent A (2 relevant calls) should score 0.8-1.0, while Agent B (irrelevant + duplicate calls) should score 0.2-0.5.

In [ ]:
from strands_evals.evaluators import TrajectoryEvaluator

traj_eval = TrajectoryEvaluator(
    rubric=(
        "Rate the tool usage trajectory 0-1:\n"
        "- 0.8-1.0: Only relevant tools called, no duplicates, logical order\n"
        "- 0.5-0.7: Mostly correct but minor inefficiency\n"
        "- 0.2-0.4: Irrelevant tools called or excessive duplicates\n"
        "- 0.0-0.1: Completely wrong tool selection"
    ),
    model=MODEL,
)

traj_cases = [
    Case(name="agent_a", input=QUESTION, expected_trajectory=["search_flights", "get_weather"]),
    Case(name="agent_b", input=QUESTION, expected_trajectory=["search_flights", "get_weather"]),
]

# Now each returns DIFFERENT trajectories with the SAME output
def traj_task(case):
    trajectory = TRAJECTORY_A if case.name == "agent_a" else TRAJECTORY_B
    return {"output": SAME_ANSWER, "trajectory": trajectory}

print("=" * 60)
print("TEST 2: TRAJECTORY — Can it tell Agent A from Agent B?")
print("=" * 60)

traj_exp = Experiment(cases=traj_cases, evaluators=[traj_eval])
traj_reports = traj_exp.run_evaluations(traj_task)
traj_reports[0].display()

print("\n✅ Trajectory evaluation distinguishes efficient from wasteful paths!")

---
## Test 3: Live agent with TrajectoryPlugin (hooks)

**What this does:** Runs a real agent (not simulated) with the `TrajectoryPlugin` hook attached. The hook captures every tool call automatically during execution — no manual trajectory construction needed.

**Why use hooks for capture:** In Tests 1 and 2, we manually constructed the trajectory lists. In production, you do not know the trajectory in advance. The `TrajectoryPlugin` hook intercepts every `AfterToolCallEvent` and records the tool name, parameters, timing, and success/failure status automatically.

> **What to look for:** The plugin display should show the complete tool call sequence, including timing for each call. Compare the live agent's trajectory to Agent A's ideal path — does the real agent make unnecessary calls, or does it follow an efficient path? The tool count and order are the key signals.

In [ ]:
from strands import Agent
from strands.models.openai import OpenAIModel
from trajectory_plugin import TrajectoryPlugin, search_flights, get_weather, book_hotel, get_currency_exchange

tracker = TrajectoryPlugin()

agent = Agent(
    model=OpenAIModel(model_id=MODEL),
    tools=[search_flights, get_weather, book_hotel, get_currency_exchange],
    hooks=[tracker],
    system_prompt="You are a travel assistant. Use tools to answer questions.",
)

print("=" * 60)
print("TEST 3: LIVE AGENT with TrajectoryPlugin hooks")
print("=" * 60)

result = agent("Find flights from NYC to London and check the weather there")
print(f"\nAgent response:\n{result}")

# The hook captured everything automatically
tracker.display()

---
## Comparison Summary

| Approach | Sees Trajectory | Catches Inefficiency | Automatic Capture | Cost |
|----------|:-:|:-:|:-:|---|
| `OutputEvaluator` | ❌ | ❌ | N/A | 1 LLM call |
| `TrajectoryEvaluator` | ✅ | ✅ | ❌ (manual) | 1 LLM call |
| `TrajectoryPlugin` + hooks | ✅ | ✅ | ✅ (automatic) | 0 (deterministic) |
| Plugin + TrajectoryEvaluator | ✅ | ✅ | ✅ | 1 LLM call |

**Recommendation:** Use `TrajectoryPlugin` to capture trajectories automatically, then pass them to `TrajectoryEvaluator` for scoring. This gives you the best of both: automatic capture + LLM-based quality assessment.

**Next:** [Demo 02 - Risk Analysis](../02-trajectory-risk-analysis/) — Score risk signals like failures, duplicates, and excessive duration.